# Import Required Libraries

In [1]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator


# Define Data Paths

In [2]:
train_dir = 'Dataset\PLD_3_Classes_256\Training'
val_dir = 'Dataset\PLD_3_Classes_256\Validation'
test_dir = 'Dataset\PLD_3_Classes_256\Testing'




# Define Image Parameters

In [1]:
width = 150  # Width of the images
height = 150  # Height of the images
num_channels = 3  # Number of color channels (RGB)

num_classes = 3  # Number of classes in your dataset
class_names = ["EarlyBlight", "Healthy", "LateBlight"]  # List of class names in order

# Your image input shape would be:
input_shape = (width, height, num_channels)

# The total number of categories/classes your model needs to predict:
total_classes = num_classes

input_shape = (width, height, num_channels)  # Adjust to match your image dimensions and channels
num_classes = len(os.listdir(train_dir))  # Assuming each class is represented by a subfolder


NameError: name 'os' is not defined

# Data Augmentation:

In [4]:
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255.0,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

val_datagen = ImageDataGenerator(rescale=1.0 / 255.0)


# Load Data Generators:

In [5]:
batch_size = 32  # Adjust as needed
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(width, height),
    batch_size=batch_size,
    class_mode='categorical'
)

val_generator = val_datagen.flow_from_directory(
    val_dir,
    target_size=(width, height),
    batch_size=batch_size,
    class_mode='categorical'
)


Found 3251 images belonging to 3 classes.
Found 416 images belonging to 3 classes.


# Build the CNN Model:

In [6]:
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=input_shape),
    MaxPooling2D(2, 2),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),
    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),
    Flatten(),
    Dense(512, activation='relu'),
    Dropout(0.5),
    Dense(num_classes, activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])


# Train the Model:

In [7]:
epochs = 10  # Adjust as needed
history = model.fit(
    train_generator,
    steps_per_epoch=train_generator.samples // batch_size,
    epochs=epochs,
    validation_data=val_generator,
    validation_steps=val_generator.samples // batch_size
)


Epoch 1/10
101/101 [==============================] - 74s 725ms/step - loss: 1.0572 - accuracy: 0.4427 - val_loss: 1.0063 - val_accuracy: 0.4183
Epoch 2/10
101/101 [==============================] - 76s 746ms/step - loss: 0.9587 - accuracy: 0.5517 - val_loss: 0.8567 - val_accuracy: 0.5673
Epoch 3/10
101/101 [==============================] - 76s 747ms/step - loss: 0.7630 - accuracy: 0.6741 - val_loss: 0.7446 - val_accuracy: 0.7115
Epoch 4/10
101/101 [==============================] - 77s 763ms/step - loss: 0.6033 - accuracy: 0.7623 - val_loss: 0.4298 - val_accuracy: 0.8582
Epoch 5/10
101/101 [==============================] - 78s 776ms/step - loss: 0.5270 - accuracy: 0.7928 - val_loss: 0.5422 - val_accuracy: 0.7572
Epoch 6/10
101/101 [==============================] - 79s 783ms/step - loss: 0.5138 - accuracy: 0.8105 - val_loss: 0.4298 - val_accuracy: 0.8221
Epoch 7/10
101/101 [==============================] - 77s 763ms/step - loss: 0.4189 - accuracy: 0.8484 - val_loss: 0.3919 - val_ac

# Evaluate the Model:

In [8]:
test_generator = val_datagen.flow_from_directory(
    test_dir,
    target_size=(width, height),
    batch_size=batch_size,
    class_mode='categorical'
)

test_loss, test_acc = model.evaluate(test_generator)
print("Test accuracy:", test_acc)


Found 405 images belonging to 3 classes.
13/13 [==============================] - 2s 155ms/step - loss: 0.3016 - accuracy: 0.9012
Test accuracy: 0.9012345671653748


# Make Predictions

In [11]:
from PIL import Image

# Load and preprocess a new image
from PIL import Image
new_image = Image.open('Healthy_2.jpg')
new_image = new_image.resize((width, height))
new_image = np.array(new_image) / 255.0

# Make predictions
predictions = model.predict(np.expand_dims(new_image, axis=0))
predicted_class_index = np.argmax(predictions[0])

# Get class labels from the generator
class_labels = list(train_generator.class_indices.keys())

# Map the predicted class index to the class label
predicted_class_label = class_labels[predicted_class_index]

print("Predicted class:", predicted_class_label)

1/1 [==============================] - 0s 36ms/step
Predicted class: Healthy


# KNN

In [12]:
from sklearn.metrics import precision_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
# Load and Preprocess Images
def load_images_from_folder(folder):
    images = []
    labels = []
    for class_index, class_name in enumerate(os.listdir(folder)):
        class_path = os.path.join(folder, class_name)
        for filename in os.listdir(class_path):
            if filename.endswith('.jpg') or filename.endswith('.png'):
                img_path = os.path.join(class_path, filename)
                img = Image.open(img_path)
                img = img.resize((width, height))
                img = np.array(img) / 255.0
                images.append(img)
                labels.append(class_index)
    return np.array(images), np.array(labels)

train_images, train_labels = load_images_from_folder(train_dir)
test_images, test_labels = load_images_from_folder(test_dir)

# Flatten the images for KNN
train_images_flatten = train_images.reshape(train_images.shape[0], -1)
test_images_flatten = test_images.reshape(test_images.shape[0], -1)

# Train KNN Model
k = 5  # Number of neighbors
knn_model = KNeighborsClassifier(n_neighbors=k)
knn_model.fit(train_images_flatten, train_labels)

# Predict using KNN Model
test_predictions = knn_model.predict(test_images_flatten)

# Evaluate KNN Model
accuracy = accuracy_score(test_labels, test_predictions)
print("KNN Model Accuracy:", accuracy)
# Calculate Precision Metrics
precision = precision_score(test_labels, test_predictions, average=None)

# Print Precision for Each Class
class_names = os.listdir(train_dir)
for i, class_name in enumerate(class_names):
    print(f"Precision for {class_name}: {precision[i]}")


KNN Model Accuracy: 0.5580246913580247
Precision for Early_Blight: 0.5465587044534413
Precision for Healthy: 0.56
Precision for Late_Blight: 0.6363636363636364
